In [7]:
import torch as py
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Subset
import torchvision
import random
from torchvision.models._api import WeightsEnum

import torch
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import transforms
from PIL import Image

from sklearn.metrics import classification_report
from torch.nn import functional as F
import torch.nn as nn
import torch.optim as optim


In [9]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [11]:
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN, InceptionResnetV1
from torchvision import transforms
from PIL import Image
import os
import torch

class DeepfakeFaceDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.image_paths = []
        self.labels = []

        self.classes = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

        for label_idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)
            for file in os.listdir(class_dir):
                if file.endswith(('.jpg', '.png')):
                    self.image_paths.append(os.path.join(class_dir, file))
                    self.labels.append(label_idx)


        self.mtcnn = MTCNN(image_size=224, margin=20)
        self.feature_extractor = InceptionResnetV1(pretrained='vggface2').eval()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        img = Image.open(img_path).convert("RGB")

        face = self.mtcnn(img)
        if face is None:
            return torch.zeros(3, 224, 224), -1  # return dummy image if no face
        return face, label


In [13]:
dataset = DeepfakeFaceDataset("/Users/amrutavelamuri/Downloads/GitHubDataset/Train")
for epoch in range(10):
    subset_size = min(2000, len(dataset))
    subset_indices = random.sample(range(len(dataset)), subset_size)

    small_dataset = Subset(dataset, subset_indices)
    dataloader = DataLoader(small_dataset, batch_size=5, shuffle=True, num_workers=0)


In [15]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0)

dataloader=DataLoader(    
    dataset,batch_size=20,shuffle=True,num_workers=0)


for batch_idx, (images, labels) in enumerate(dataloader):
    if batch_idx == 5:  # After 5 batches, stop (you can adjust the number)
        break
    print(f"Batch {batch_idx + 1} labels: {labels.tolist()}")

Batch 1 labels: [1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1]
Batch 2 labels: [0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1]
Batch 3 labels: [1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1]
Batch 4 labels: [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0]
Batch 5 labels: [1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]


In [ ]:
print(f"Total images loaded: {len(dataset)}")

In [ ]:

def imshow(img):
    img = img.cpu() * 0.5 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis("off")
    plt.show()
for batch_idx, (images, labels) in enumerate(dataloader):
    if batch_idx == 1:  
        random_indices = random.sample(range(images.size(0)), 20)
        selected_images = images[random_indices]
        selected_labels = labels[random_indices]

        imshow(torchvision.utils.make_grid(selected_images))

        for idx in range(20):
            label_index = selected_labels[idx].item()
            class_name = dataset.classes[label_index]
            print(f"Label Index: {label_index} | Class Name: {class_name}")
        break


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, output_dim):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 226, 226)
            x = self.pool1(F.relu(self.conv1(dummy_input)))
            x = self.pool2(F.relu(self.conv2(x)))
            self.flattened_size = x.view(1, -1).shape[1]
        print(x.shape)
        self.fc1 = nn.Linear(100352, 10)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

In [ ]:
model = SimpleCNN(output_dim=10)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

train_losses = []
train_accuracies = []

for epoch in range(2):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:  # Use  training DataLoader here
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(dataloader)
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}")
